# LeetCode 737 — Sentence Similarity II bằng Python

Link đề: https://leetcode.com/problems/sentence-similarity-ii/ · Độ khó: **Medium** · Chủ đề: Union Find, Depth-First Search, Breadth-First Search, Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def areSentencesSimilarTwo(self, sentence1: List[str], sentence2: List[str], similarPairs: List[List[str]]) -> bool:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí để xem đề gốc). Đề trong notebook được viết lại theo hiểu biết về bài và theo các ví dụ đã biết; **nên đối chiếu lại đề gốc nếu bạn có tài khoản Premium**, đặc biệt phần ví dụ và ràng buộc.

Ý chính: quan hệ "giống nghĩa" có tính **bắc cầu** (`a ~ b` và `b ~ c` thì `a ~ c`), nên không thể chỉ tra cặp trực tiếp. Mỗi nhóm từ giống nghĩa là một **thành phần liên thông** của đồ thị. Notebook đi từ cách "tìm đường cho từng cặp từ" (DFS), qua gán nhãn thành phần bằng BFS, đến **Union-Find** (cách nên nộp), và cách gộp tập hợp nhỏ vào tập lớn.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai câu (mảng từ) giống nhau nếu **cùng độ dài** và từ thứ `i` của câu 1 **giống nghĩa** từ thứ `i` của câu 2; giống nghĩa có tính **phản xạ** (từ giống chính nó), **đối xứng** và **bắc cầu**.
2. Nắm các công cụ Python: `dict`/`set`, `defaultdict(list)` cho danh sách kề, hàng đợi `deque` cho BFS, ngăn xếp (list) cho DFS, `zip`, `all`, và cấu trúc **Union-Find** (Disjoint Set Union).
3. Đi từ **DFS cho từng cặp từ** → **gán nhãn thành phần liên thông bằng BFS** → **Union-Find với nén đường đi** → **gộp tập hợp nhỏ vào tập lớn**.
4. Hiểu mấu chốt: quan hệ giống nghĩa là **quan hệ tương đương**, nên chia các từ thành các **lớp tương đương** (thành phần liên thông); hai từ giống nghĩa khi **bằng nhau** hoặc **cùng lớp**.
5. Tránh các lỗi kinh điển: coi giống nghĩa là không bắc cầu, quên kiểm tra độ dài, quên trường hợp hai từ bằng nhau nhưng không xuất hiện trong cặp nào, so sánh `parent[x]` thay vì `find(x)`, và `find` đệ quy bị `RecursionError` với chuỗi dài.

## 2. Đề bài & ví dụ

Một câu được biểu diễn bằng mảng các từ, ví dụ `"I am happy with leetcode"` là `["I","am","happy","with","leetcode"]`.

Cho hai câu `sentence1`, `sentence2` (hai mảng chuỗi) và mảng `similarPairs` gồm các cặp `[x, y]` cho biết hai từ `x` và `y` **giống nghĩa**.

Trả về `True` nếu hai câu giống nhau, `False` nếu không. Hai câu giống nhau khi:

- Chúng có **cùng độ dài** (cùng số từ);
- Với mọi `i`, `sentence1[i]` và `sentence2[i]` **giống nghĩa**.

Chú ý: một từ **luôn giống nghĩa với chính nó** (kể cả khi nó không có trong `similarPairs`), và quan hệ giống nghĩa có tính **bắc cầu**: nếu `a ~ b` và `b ~ c` thì `a ~ c`. Quan hệ cũng đối xứng (cặp `[x, y]` cho cả `x ~ y` và `y ~ x`).

**Ví dụ 1**

- Input: `sentence1 = ["great","acting","skills"]`, `sentence2 = ["fine","drama","talent"]`, `similarPairs = [["great","good"],["fine","good"],["drama","acting"],["skills","talent"]]`
- Output: `True`
- Giải thích: `great ~ good ~ fine`, `drama ~ acting`, `skills ~ talent`.

**Ví dụ 2**

- Input: `sentence1 = ["I","love","leetcode"]`, `sentence2 = ["I","love","onepiece"]`, `similarPairs = [["manga","onepiece"],["platform","anime"],["leetcode","platform"],["anime","manga"]]`
- Output: `True`
- Giải thích: `leetcode -> platform -> anime -> manga -> onepiece`, hai từ đầu bằng nhau.

**Ví dụ 3**

- Input: `sentence1 = ["I","love","leetcode"]`, `sentence2 = ["I","love","onepiece"]`, `similarPairs = [["manga","hunterXhunter"],["platform","anime"],["leetcode","platform"],["anime","manga"]]`
- Output: `False`
- Giải thích: `leetcode` không giống nghĩa `onepiece`.

**Ràng buộc:**

- `1 <= sentence1.length, sentence2.length <= 1000`
- `1 <= sentence1[i].length, sentence2[i].length <= 20`, chỉ gồm chữ cái Anh (hoa và thường, **phân biệt hoa/thường**)
- `0 <= similarPairs.length <= 2000`
- `similarPairs[i].length == 2`, `1 <= xi.length, yi.length <= 20`, chỉ gồm chữ cái Anh

Hệ quả: số từ phân biệt trong `similarPairs` tối đa `4000`; số cặp so sánh tối đa `1000`.

In [1]:
from typing import List
from collections import defaultdict, deque
import random
import time
import sys

examples = [
    ((["great", "acting", "skills"], ["fine", "drama", "talent"],
      [["great", "good"], ["fine", "good"], ["drama", "acting"], ["skills", "talent"]]), True),
    ((["I", "love", "leetcode"], ["I", "love", "onepiece"],
      [["manga", "onepiece"], ["platform", "anime"], ["leetcode", "platform"], ["anime", "manga"]]), True),
    ((["I", "love", "leetcode"], ["I", "love", "onepiece"],
      [["manga", "hunterXhunter"], ["platform", "anime"], ["leetcode", "platform"], ["anime", "manga"]]), False),
]
for (s1, s2, pairs), expected in examples:
    print(len(s1), len(s2), "| số cặp:", len(pairs), "| kỳ vọng:", expected)

3 3 | số cặp: 4 | kỳ vọng: True
3 3 | số cặp: 4 | kỳ vọng: True
3 3 | số cặp: 4 | kỳ vọng: False


## 3. Kiến thức nền

### 3.1. Đồ thị, danh sách kề và thành phần liên thông

Ta xem **mỗi từ là một đỉnh**, mỗi cặp `[x, y]` là một **cạnh** nối `x` và `y` (vô hướng). Hai từ giống nghĩa (bắc cầu) khi có một **đường đi** nối chúng, tức khi chúng nằm trong cùng một **thành phần liên thông**. **Danh sách kề** `adj[x]` chứa các đỉnh kề với `x`; `defaultdict(list)` tự tạo danh sách rỗng khi truy cập khóa mới.

In [2]:
pairs = [["manga", "onepiece"], ["platform", "anime"], ["leetcode", "platform"], ["anime", "manga"]]
adj = defaultdict(list)
for a, b in pairs:
    adj[a].append(b)
    adj[b].append(a)           # vô hướng: thêm cả hai chiều
for word, nbrs in adj.items():
    print(word, "->", nbrs)

manga -> ['onepiece', 'anime']
onepiece -> ['manga']
platform -> ['anime', 'leetcode']
anime -> ['platform', 'manga']
leetcode -> ['platform']


### 3.2. DFS và BFS: tìm đường đi

**DFS** (duyệt theo chiều sâu) dùng **ngăn xếp**: lấy đỉnh trên cùng, đẩy các đỉnh kề chưa thăm vào. **BFS** (duyệt theo chiều rộng) dùng **hàng đợi** `deque`: lấy đỉnh đầu hàng, đẩy kề vào cuối. Cả hai đánh dấu đỉnh đã thăm bằng `set` để không lặp vô hạn. Duyệt từ `start` thăm được **đúng các đỉnh cùng thành phần** với `start`.

In [3]:
def reachable(adj, start):
    seen = {start}
    stack = [start]                  # DFS bằng ngăn xếp (list)
    while stack:
        u = stack.pop()
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    return seen

print(sorted(reachable(adj, "leetcode")))
queue = deque(["leetcode"])
seen = {"leetcode"}
while queue:                          # BFS bằng deque
    u = queue.popleft()
    for v in adj[u]:
        if v not in seen:
            seen.add(v)
            queue.append(v)
print(sorted(seen) == sorted(reachable(adj, "leetcode")))

['anime', 'leetcode', 'manga', 'onepiece', 'platform']
True


### 3.3. Union-Find (Disjoint Set Union)

Union-Find quản lý **các tập rời nhau**. Mỗi phần tử có một **cha** `parent`; gốc của cây là phần tử có `parent[x] == x` và đại diện cho tập. Hai thao tác:

- `find(x)`: đi lên theo `parent` đến gốc. **Nén đường đi** (path compression): gắn thẳng các nút trên đường vào gốc để lần sau nhanh.
- `union(a, b)`: nối gốc của `a` với gốc của `b` (nếu khác nhau). **Hợp theo kích thước** (union by size): gắn cây nhỏ vào cây lớn.

Với cả hai tối ưu, mỗi thao tác gần như `O(1)` (hàm ngược Ackermann).

In [4]:
parent = list(range(6))

def find(x):
    root = x
    while parent[root] != root:
        root = parent[root]
    while parent[x] != root:          # nén đường đi (lặp, không đệ quy)
        parent[x], x = root, parent[x]
    return root

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

union(0, 1); union(1, 2); union(4, 5)
print([find(i) for i in range(6)])
print(find(0) == find(2), find(0) == find(4))

[0, 0, 0, 3, 4, 4]
True False


### 3.4. `dict`, `set` và so sánh chuỗi

`words = {}` ánh xạ từ sang số thứ tự; `x in words` kiểm tra có mặt trong `O(1)`. Chuỗi trong Python so sánh bằng `==` **phân biệt hoa/thường** (`"A" != "a"`). Từ nào **không có** trong mọi cặp thì chỉ giống nghĩa với **chính nó**.

In [5]:
words = {}
for w in ["great", "good", "great", "fine"]:
    if w not in words:
        words[w] = len(words)
print(words)
print("A" == "a", "great" == "great")
print("unknown" in words)

{'great': 0, 'good': 1, 'fine': 2}
False True
False


### 3.5. `zip` và `all`

`zip(s1, s2)` ghép từng cặp phần tử cùng vị trí của hai danh sách (dừng ở danh sách ngắn hơn, nên cần kiểm tra độ dài trước). `all(cond for ...)` đúng khi mọi điều kiện đúng.

In [6]:
s1 = ["I", "love", "it"]
s2 = ["I", "like", "it"]
print(list(zip(s1, s2)))
print(all(a == b for a, b in zip(s1, s2)), all(len(a) > 0 for a in s1))
print(len(s1) == len(s2))

[('I', 'I'), ('love', 'like'), ('it', 'it')]
False True
True


## 4. Ý tưởng nền tảng / trực giác

Xét ví dụ 2. Các cặp tạo thành đồ thị:

```
leetcode -- platform -- anime -- manga -- onepiece
```

`leetcode` và `onepiece` **không** có cặp trực tiếp, nhưng nối với nhau qua chuỗi `platform, anime, manga`: nên giống nghĩa nhờ **bắc cầu**. Nếu ta chỉ tra "có cặp trực tiếp hay không" (như bài Sentence Similarity I), ta sẽ trả sai `False`.

Hãy thử tay: với mỗi cặp từ cần so sánh, "đi" theo các cặp để xem có tới được từ kia không.

In [7]:
(s1, s2, pairs), expected = examples[1]
adj = defaultdict(list)
for a, b in pairs:
    adj[a].append(b)
    adj[b].append(a)

for a, b in zip(s1, s2):
    if a == b:
        print(f"{a!r} == {b!r}: bằng nhau, giống nghĩa (phản xạ)")
    else:
        comp = reachable(adj, a)
        print(f"{a!r} vs {b!r}: các từ đến được từ {a!r} = {sorted(comp)} -> {b in comp}")

'I' == 'I': bằng nhau, giống nghĩa (phản xạ)
'love' == 'love': bằng nhau, giống nghĩa (phản xạ)
'leetcode' vs 'onepiece': các từ đến được từ 'leetcode' = ['anime', 'leetcode', 'manga', 'onepiece', 'platform'] -> True


Quan sát:

- **Từ bằng nhau** (`I`, `love`) giống nghĩa nhau không cần cặp nào.
- Với từ khác nhau, hai từ giống nghĩa khi chúng **cùng thành phần liên thông** của đồ thị từ các cặp.
- Mọi từ trong một thành phần liên thông đều giống nghĩa **đôi một** (vì đi qua chuỗi cạnh). Vậy bài toán là: **chia các từ thành các nhóm liên thông, rồi so từng vị trí**.

Mục 5 làm cách trực tiếp nhất (tìm đường cho từng cặp từ), mục 6 và 7 làm gọn hơn nhiều.

## 5. Cách 1 — Brute Force: DFS tìm đường cho từng cặp từ

**Ý tưởng.** Dựng danh sách kề. Với mỗi vị trí `i`: nếu hai từ bằng nhau thì qua; ngược lại chạy một lần **DFS** từ `sentence1[i]` xem có đến được `sentence2[i]` không (nếu không → `False`). Dùng DFS lặp (ngăn xếp) để không bị `RecursionError`.

In [8]:
def similar_dfs(sentence1, sentence2, pairs):
    if len(sentence1) != len(sentence2):
        return False
    adj = defaultdict(list)
    for a, b in pairs:
        adj[a].append(b)
        adj[b].append(a)
    for a, b in zip(sentence1, sentence2):
        if a == b:
            continue
        seen = {a}
        stack = [a]
        found = False
        while stack and not found:
            u = stack.pop()
            for v in adj.get(u, ()):
                if v == b:
                    found = True
                    break
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
        if not found:
            return False
    return True


for (s1, s2, pairs), expected in examples:
    got = similar_dfs(s1, s2, pairs)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

True (kỳ vọng True )
True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp:** Time `O(n · (V + E))` với `n` là số vị trí so sánh (≤ 1000), `V` số từ phân biệt (≤ 4000), `E` số cặp (≤ 2000): tối đa khoảng `1000 × 6000 = 6·10^6` thao tác: vẫn qua giới hạn đề nhưng **lặp lại công việc**. Space `O(V + E)`.

**Nút thắt:** mỗi lần DFS ta duyệt lại cả thành phần liên thông, nhiều vị trí cùng chạm vào cùng thành phần. Ta chỉ cần biết "từ nào thuộc nhóm nào" một lần duy nhất.

## 6. Ý tưởng thuật toán / bước đột phá

Quan hệ "giống nghĩa" (kể cả tự thân) là **quan hệ tương đương**: phản xạ, đối xứng, bắc cầu. Mỗi quan hệ tương đương chia tập từ thành các **lớp tương đương** rời nhau, chính là **thành phần liên thông** của đồ thị các cặp. Hai từ giống nghĩa **khi và chỉ khi** cùng lớp.

Vì vậy làm **một lần**: gán cho mỗi từ **mã nhóm** (nhãn thành phần). Sau đó mỗi vị trí chỉ là so hai mã nhóm: `O(1)`.

Có hai cách chính để có mã nhóm:

- **Duyệt đồ thị** (BFS/DFS) từ mọi đỉnh chưa gán nhãn, gán chung một nhãn cho cả thành phần: `O(V + E)`.
- **Union-Find**: mỗi cặp `[x, y]` thực hiện `union(x, y)`; mã nhóm của từ là `find(từ)`. Rất hợp với bài "gộp nhóm dần dần", và không cần dựng đồ thị.

Quy tắc so sánh cho một vị trí: `a == b` hoặc (`a` và `b` đều có mặt trong các cặp và cùng nhóm). Một từ không có trong cặp nào chỉ giống nghĩa với chính nó.

## 7. Các cách tiếp theo

Phần này có thêm 3 cách nữa (tổng 4 cách): gán nhãn thành phần bằng BFS, Union-Find (cách nên nộp), và gộp tập hợp nhỏ vào tập lớn.

### 7.1. Cách 2 — Gán nhãn thành phần liên thông bằng BFS (một lần)

**Giải thích từng đoạn.**

1. Dựng danh sách kề từ các cặp.
2. `comp = {}` ánh xạ từ → mã nhóm. Với mỗi từ `w` trong danh sách kề mà chưa có nhãn, chạy BFS từ `w` và gán **cùng một mã** `cid` cho mọi từ thăm được; sau đó `cid += 1`.
3. So sánh từng vị trí: `a == b`, hoặc `a`, `b` đều có trong `comp` và `comp[a] == comp[b]`.

In [9]:
def similar_bfs_label(sentence1, sentence2, pairs):
    if len(sentence1) != len(sentence2):
        return False
    adj = defaultdict(list)
    for a, b in pairs:
        adj[a].append(b)
        adj[b].append(a)
    comp = {}
    cid = 0
    for start in adj:
        if start in comp:
            continue
        comp[start] = cid
        queue = deque([start])
        while queue:
            u = queue.popleft()
            for v in adj[u]:
                if v not in comp:
                    comp[v] = cid
                    queue.append(v)
        cid += 1
    return all(a == b or (a in comp and b in comp and comp[a] == comp[b])
               for a, b in zip(sentence1, sentence2))


for (s1, s2, pairs), expected in examples:
    got = similar_bfs_label(s1, s2, pairs)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

True (kỳ vọng True )
True (kỳ vọng True )
False (kỳ vọng False )


In [10]:
class Solution:
    def areSentencesSimilarTwo(self, sentence1: List[str], sentence2: List[str], similarPairs: List[List[str]]) -> bool:
        if len(sentence1) != len(sentence2):
            return False
        adj = defaultdict(list)
        for a, b in similarPairs:
            adj[a].append(b)
            adj[b].append(a)
        comp = {}
        cid = 0
        for start in adj:
            if start in comp:
                continue
            comp[start] = cid
            queue = deque([start])
            while queue:
                u = queue.popleft()
                for v in adj[u]:
                    if v not in comp:
                        comp[v] = cid
                        queue.append(v)
            cid += 1
        return all(a == b or (a in comp and b in comp and comp[a] == comp[b])
                   for a, b in zip(sentence1, sentence2))

print(Solution().areSentencesSimilarTwo(*examples[1][0]))

True


**Độ phức tạp:** Time `O(V + E + n)` (mỗi đỉnh và cạnh duyệt đúng một lần, rồi `n` phép so sánh `O(1)`); mỗi từ có độ dài ≤ 20 nên băm chuỗi coi như `O(1)`. Space `O(V + E)` cho đồ thị và nhãn.

### 7.2. Cách 3 — Union-Find (cách nên nộp)

**Giải thích từng đoạn.**

1. Gán mỗi từ phân biệt một số thứ tự (`ids[w]`) khi gặp lần đầu trong các cặp.
2. `parent[i] = i`, `size[i] = 1`. Với mỗi cặp `[x, y]` gọi `union(ids[x], ids[y])`.
3. `find` viết **lặp** (không đệ quy) kèm nén đường đi. Lý do: chuỗi 2000 cặp có thể tạo cây rất sâu; `find` đệ quy sẽ vượt giới hạn đệ quy mặc định của Python (1000) và gây `RecursionError`.
4. So sánh từng vị trí: `a == b`, hoặc cả hai trong `ids` và `find(ids[a]) == find(ids[b])`.

In [11]:
def similar_union_find(sentence1, sentence2, pairs):
    if len(sentence1) != len(sentence2):
        return False
    ids = {}
    for a, b in pairs:
        for w in (a, b):
            if w not in ids:
                ids[w] = len(ids)
    parent = list(range(len(ids)))
    size = [1] * len(ids)

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:               # nén đường đi
            parent[x], x = root, parent[x]
        return root

    for a, b in pairs:
        ra, rb = find(ids[a]), find(ids[b])
        if ra != rb:
            if size[ra] < size[rb]:
                ra, rb = rb, ra
            parent[rb] = ra                    # gắn cây nhỏ vào cây lớn
            size[ra] += size[rb]

    return all(a == b or (a in ids and b in ids and find(ids[a]) == find(ids[b]))
               for a, b in zip(sentence1, sentence2))


for (s1, s2, pairs), expected in examples:
    got = similar_union_find(s1, s2, pairs)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

True (kỳ vọng True )
True (kỳ vọng True )
False (kỳ vọng False )


In [12]:
class Solution:
    def areSentencesSimilarTwo(self, sentence1: List[str], sentence2: List[str], similarPairs: List[List[str]]) -> bool:
        if len(sentence1) != len(sentence2):
            return False
        ids = {}
        for a, b in similarPairs:
            for w in (a, b):
                if w not in ids:
                    ids[w] = len(ids)
        parent = list(range(len(ids)))
        size = [1] * len(ids)

        def find(x):
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:
                parent[x], x = root, parent[x]
            return root

        for a, b in similarPairs:
            ra, rb = find(ids[a]), find(ids[b])
            if ra != rb:
                if size[ra] < size[rb]:
                    ra, rb = rb, ra
                parent[rb] = ra
                size[ra] += size[rb]

        return all(a == b or (a in ids and b in ids and find(ids[a]) == find(ids[b]))
                   for a, b in zip(sentence1, sentence2))

print(Solution().areSentencesSimilarTwo(*examples[0][0]))

True


**Độ phức tạp:** Time `O((P + n) · α(V))` với `P` số cặp, `α` hàm ngược Ackermann (≤ 4 trong thực tế), tức coi như `O(P + n)`. Space `O(V)` cho `ids`, `parent`, `size`.

### 7.3. Cách 4 — Gộp tập hợp nhỏ vào tập lớn (small-to-large)

**Giải thích từng đoạn.** Mỗi từ trỏ tới **tập** (`set`) chứa mọi từ cùng nhóm. Khi gặp cặp `[x, y]`:

- cả hai chưa có tập → tạo tập mới `{x, y}`;
- một trong hai có tập → thêm từ còn lại vào tập đó;
- cả hai có tập khác nhau → **gộp tập nhỏ vào tập lớn** (duyệt các phần tử của tập nhỏ, đổi con trỏ của chúng sang tập lớn). Mỗi phần tử bị chuyển nhiều nhất `log V` lần (mỗi lần chuyển, tập chứa nó **ít nhất gấp đôi**).

Hai từ cùng nhóm khi chúng **trỏ cùng một đối tượng tập** (`is`). Không cần cấu trúc Union-Find.

In [13]:
def similar_merge_sets(sentence1, sentence2, pairs):
    if len(sentence1) != len(sentence2):
        return False
    group = {}                                   # từ -> tập chứa nó
    for a, b in pairs:
        ga, gb = group.get(a), group.get(b)
        if ga is None and gb is None:
            g = {a, b}
            group[a] = group[b] = g
        elif ga is None:
            gb.add(a)
            group[a] = gb
        elif gb is None:
            ga.add(b)
            group[b] = ga
        elif ga is not gb:
            if len(ga) < len(gb):
                ga, gb = gb, ga                  # ga là tập lớn
            for w in gb:                         # chuyển tập nhỏ sang tập lớn
                group[w] = ga
            ga |= gb
    return all(a == b or (a in group and b in group and group[a] is group[b])
               for a, b in zip(sentence1, sentence2))


for (s1, s2, pairs), expected in examples:
    got = similar_merge_sets(s1, s2, pairs)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

True (kỳ vọng True )
True (kỳ vọng True )
False (kỳ vọng False )


In [14]:
class Solution:
    def areSentencesSimilarTwo(self, sentence1: List[str], sentence2: List[str], similarPairs: List[List[str]]) -> bool:
        if len(sentence1) != len(sentence2):
            return False
        group = {}
        for a, b in similarPairs:
            ga, gb = group.get(a), group.get(b)
            if ga is None and gb is None:
                group[a] = group[b] = {a, b}
            elif ga is None:
                gb.add(a)
                group[a] = gb
            elif gb is None:
                ga.add(b)
                group[b] = ga
            elif ga is not gb:
                if len(ga) < len(gb):
                    ga, gb = gb, ga
                for w in gb:
                    group[w] = ga
                ga |= gb
        return all(a == b or (a in group and b in group and group[a] is group[b])
                   for a, b in zip(sentence1, sentence2))

print(Solution().areSentencesSimilarTwo(*examples[2][0]))

False


**Độ phức tạp:** Time `O(P log V + n)` (mỗi phần tử đổi tập tối đa `log V` lần). Space `O(V)`. Dễ viết, không cần `find`; nhược điểm là chậm hơn Union-Find một hệ số `log` trong trường hợp xấu.

## 8. TRACE từng bước

Trace Union-Find (cách 3) với ví dụ 1. Các từ phân biệt theo thứ tự gặp: `great=0, good=1, fine=2, drama=3, acting=4, skills=5, talent=6`. Ban đầu `parent = [0,1,2,3,4,5,6]`, `size = [1,1,1,1,1,1,1]`.

| cặp | `find` hai đầu | hành động | `parent` sau | `size` của gốc |
|---|---|---|---|---|
| `great, good` | 0, 1 | khác gốc; kích thước bằng nhau, gắn 1 vào 0 | `[0,0,2,3,4,5,6]` | `size[0] = 2` |
| `fine, good` | 2, 0 | khác gốc; `size[2]=1 < size[0]=2`, gắn 2 vào 0 | `[0,0,0,3,4,5,6]` | `size[0] = 3` |
| `drama, acting` | 3, 4 | gắn 4 vào 3 | `[0,0,0,3,3,5,6]` | `size[3] = 2` |
| `skills, talent` | 5, 6 | gắn 6 vào 5 | `[0,0,0,3,3,5,5]` | `size[5] = 2` |

So sánh: `great`(0) và `fine`(2): `find` = 0 và 0 → giống nghĩa. `acting`(4) và `drama`(3): gốc 3 và 3. `skills`(5) và `talent`(6): gốc 5 và 5. Cả ba cặp đều cùng nhóm → `True`.

In [15]:
def trace_union_find(sentence1, sentence2, pairs):
    ids = {}
    for a, b in pairs:
        for w in (a, b):
            if w not in ids:
                ids[w] = len(ids)
    print("ids =", ids)
    parent = list(range(len(ids)))
    size = [1] * len(ids)

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    for a, b in pairs:
        ra, rb = find(ids[a]), find(ids[b])
        if ra == rb:
            print(f"union({a},{b}): cùng gốc {ra}, bỏ qua")
            continue
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]
        print(f"union({a},{b}): gắn gốc {rb} vào gốc {ra}; parent={parent}; size[{ra}]={size[ra]}")
    ok = True
    for a, b in zip(sentence1, sentence2):
        if a == b:
            print(f"so sánh {a!r} và {b!r}: bằng nhau")
        elif a in ids and b in ids:
            ra, rb = find(ids[a]), find(ids[b])
            print(f"so sánh {a!r} và {b!r}: gốc {ra} và {rb} -> {ra == rb}")
            ok = ok and ra == rb
        else:
            print(f"so sánh {a!r} và {b!r}: có từ không nằm trong cặp nào -> False")
            ok = False
    print("kết quả:", ok)
    return ok

assert trace_union_find(*examples[0][0]) is True

ids = {'great': 0, 'good': 1, 'fine': 2, 'drama': 3, 'acting': 4, 'skills': 5, 'talent': 6}
union(great,good): gắn gốc 1 vào gốc 0; parent=[0, 0, 2, 3, 4, 5, 6]; size[0]=2
union(fine,good): gắn gốc 2 vào gốc 0; parent=[0, 0, 0, 3, 4, 5, 6]; size[0]=3
union(drama,acting): gắn gốc 4 vào gốc 3; parent=[0, 0, 0, 3, 3, 5, 6]; size[3]=2
union(skills,talent): gắn gốc 6 vào gốc 5; parent=[0, 0, 0, 3, 3, 5, 5]; size[5]=2
so sánh 'great' và 'fine': gốc 0 và 0 -> True
so sánh 'acting' và 'drama': gốc 3 và 3 -> True
so sánh 'skills' và 'talent': gốc 5 và 5 -> True
kết quả: True


Trace ví dụ 3 (kỳ vọng `False`): các cặp `manga-hunterXhunter`, `platform-anime`, `leetcode-platform`, `anime-manga` gộp thành nhóm `{manga, hunterXhunter, platform, anime, leetcode}`; `onepiece` **không có trong cặp nào** nên không thuộc nhóm. So `leetcode` và `onepiece` → `onepiece` không có trong `ids` → `False`.

In [16]:
assert trace_union_find(*examples[2][0]) is False
print()
assert trace_union_find(*examples[1][0]) is True

ids = {'manga': 0, 'hunterXhunter': 1, 'platform': 2, 'anime': 3, 'leetcode': 4}
union(manga,hunterXhunter): gắn gốc 1 vào gốc 0; parent=[0, 0, 2, 3, 4]; size[0]=2
union(platform,anime): gắn gốc 3 vào gốc 2; parent=[0, 0, 2, 2, 4]; size[2]=2
union(leetcode,platform): gắn gốc 4 vào gốc 2; parent=[0, 0, 2, 2, 2]; size[2]=3
union(anime,manga): gắn gốc 0 vào gốc 2; parent=[2, 0, 2, 2, 2]; size[2]=5
so sánh 'I' và 'I': bằng nhau
so sánh 'love' và 'love': bằng nhau
so sánh 'leetcode' và 'onepiece': có từ không nằm trong cặp nào -> False
kết quả: False

ids = {'manga': 0, 'onepiece': 1, 'platform': 2, 'anime': 3, 'leetcode': 4}
union(manga,onepiece): gắn gốc 1 vào gốc 0; parent=[0, 0, 2, 3, 4]; size[0]=2
union(platform,anime): gắn gốc 3 vào gốc 2; parent=[0, 0, 2, 2, 4]; size[2]=2
union(leetcode,platform): gắn gốc 4 vào gốc 2; parent=[0, 0, 2, 2, 2]; size[2]=3
union(anime,manga): gắn gốc 0 vào gốc 2; parent=[2, 0, 2, 2, 2]; size[2]=5
so sánh 'I' và 'I': bằng nhau
so sánh 'love' và 'love': bằn

## 9. Code nộp LeetCode

Lời giải Union-Find (Cách 3), `find` viết lặp để tránh `RecursionError`:

In [17]:
from typing import List


class Solution:
    def areSentencesSimilarTwo(
        self, sentence1: List[str], sentence2: List[str], similarPairs: List[List[str]]
    ) -> bool:
        if len(sentence1) != len(sentence2):
            return False
        ids = {}                                    # từ -> số thứ tự
        for a, b in similarPairs:
            for w in (a, b):
                if w not in ids:
                    ids[w] = len(ids)
        parent = list(range(len(ids)))
        size = [1] * len(ids)

        def find(x):
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:                # nén đường đi
                parent[x], x = root, parent[x]
            return root

        for a, b in similarPairs:
            ra, rb = find(ids[a]), find(ids[b])
            if ra != rb:
                if size[ra] < size[rb]:
                    ra, rb = rb, ra
                parent[rb] = ra
                size[ra] += size[rb]

        return all(
            a == b or (a in ids and b in ids and find(ids[a]) == find(ids[b]))
            for a, b in zip(sentence1, sentence2)
        )


sol = Solution()
for (s1, s2, pairs), expected in examples:
    got = sol.areSentencesSimilarTwo(s1, s2, pairs)
    print(got)
    assert got == expected

True
True
False


## 10. Vì sao đúng

**Bổ đề 1 — giống nghĩa là quan hệ tương đương.** Đề quy định: một từ giống chính nó (phản xạ); cặp `[x, y]` cho `x ~ y` (và `y ~ x`: đối xứng); bắc cầu. Gọi `~` là bao đóng nhỏ nhất thỏa ba tính chất chứa các cặp đã cho. Khi đó `a ~ b` khi và chỉ khi `a = b` hoặc tồn tại dãy từ `a = w0, w1, ..., wk = b` với mỗi `(wi, wi+1)` là một cặp trong `similarPairs` (theo một chiều nào đó). Nghĩa là `a` và `b` nằm trong cùng **thành phần liên thông** của đồ thị các cặp.

**Bổ đề 2 — Union-Find giữ đúng các thành phần liên thông.** Bất biến: sau khi xử lý một số cặp đầu tiên, hai từ có cùng `find` khi và chỉ khi chúng liên thông bằng các cặp đã xử lý. Ban đầu mỗi từ riêng một tập (đúng). Khi xử lý cặp `[x, y]`: nếu `find(x) != find(y)`, gộp hai tập, nên mọi từ của hai thành phần cũ trở thành liên thông (qua cạnh `x-y`), đúng; nếu cùng gốc, không thêm liên thông mới. Các cách 2 và 4 duy trì cùng bất biến bằng gán nhãn BFS và gộp tập.

**Bổ đề 3 — từ không có trong cặp nào.** Từ như vậy là một thành phần chỉ có chính nó, nên chỉ giống nghĩa với chính nó. Kiểm tra `a == b` xử lý trường hợp này; nếu `a != b` mà một trong hai không có trong `ids` thì chắc chắn không giống nghĩa (nhánh `a in ids and b in ids` sai → `False`).

**Bổ đề 4 — độ dài.** Hai câu giống nhau cần cùng độ dài theo đề; `zip` cắt theo câu ngắn hơn nên phải kiểm tra `len` trước.

**Kết luận.** Hai câu giống nhau khi và chỉ khi cùng độ dài và với mọi vị trí `i`: `s1[i] = s2[i]` hoặc cùng thành phần liên thông. Thuật toán kiểm tra đúng điều kiện này.

In [18]:
# Kiểm chứng Bổ đề 1/2 bằng thực nghiệm: Union-Find và "đóng bao bắc cầu" (Warshall) cho cùng quan hệ
def closure_relation(pairs, vocab):
    idx = {w: i for i, w in enumerate(vocab)}
    n = len(vocab)
    rel = [[i == j for j in range(n)] for i in range(n)]     # phản xạ
    for a, b in pairs:
        rel[idx[a]][idx[b]] = rel[idx[b]][idx[a]] = True       # đối xứng
    for k in range(n):                                          # bắc cầu (Warshall)
        for i in range(n):
            if rel[i][k]:
                for j in range(n):
                    if rel[k][j]:
                        rel[i][j] = True
    return idx, rel

rng = random.Random(10)
for _ in range(300):
    vocab = [chr(97 + i) for i in range(rng.randint(2, 7))]
    pairs = [[rng.choice(vocab), rng.choice(vocab)] for _ in range(rng.randint(0, 6))]
    idx, rel = closure_relation(pairs, vocab)
    for a in vocab:
        for b in vocab:
            assert similar_union_find([a], [b], pairs) == rel[idx[a]][idx[b]], (pairs, a, b)
print("Union-Find khớp bao đóng bắc cầu trên 300 đồ thị ngẫu nhiên")

Union-Find khớp bao đóng bắc cầu trên 300 đồ thị ngẫu nhiên


## 11. Time / Space Complexity

Đặt `n` là độ dài câu (≤ 1000), `P` là số cặp (≤ 2000), `V` là số từ phân biệt trong các cặp (≤ 4000).

| Cách | Time | Space |
|---|---|---|
| 1. DFS cho từng cặp từ | `O(n · (V + P))` | `O(V + P)` |
| 2. Gán nhãn BFS | `O(V + P + n)` | `O(V + P)` |
| 3. Union-Find | `O((P + n) · α(V))` | `O(V)` |
| 4. Gộp tập nhỏ vào lớn | `O(P log V + n)` | `O(V)` |

(Độ dài từ ≤ 20 nên băm và so sánh chuỗi coi như `O(1)`.) Mọi cách ngoài cách 1 đều gần tuyến tính; cách 1 vẫn chạy được ở giới hạn đề nhưng lãng phí.

In [19]:
# Đo thực nghiệm ở kích thước tối đa: 1000 từ mỗi câu, 2000 cặp, 4000 từ phân biệt
rng = random.Random(737)
vocab = ["w%d" % i for i in range(4000)]
pairs_big = [[vocab[i], vocab[i + 1]] for i in range(0, 2000)]           # chuỗi dài: tệ nhất cho DFS/find đệ quy
s1_big = [vocab[rng.randint(0, 2000)] for _ in range(1000)]
s2_big = [vocab[rng.randint(0, 2000)] for _ in range(1000)]
solvers_perf = {"dfs": similar_dfs, "bfs_label": similar_bfs_label,
                "union_find": similar_union_find, "merge_sets": similar_merge_sets}
answers = {}
for name, f in solvers_perf.items():
    t0 = time.perf_counter()
    answers[name] = f(s1_big, s2_big, pairs_big)
    assert time.perf_counter() - t0 < 20
print(answers)
assert len(set(answers.values())) == 1

{'dfs': True, 'bfs_label': True, 'union_find': True, 'merge_sets': True}


## 12. Bộ test

Chia 4 nhóm: (a) ví dụ đề và edge case; (b) phản ví dụ cho các heuristic sai; (c) đối chiếu mọi cách với **oracle độc lập** (bao đóng bắc cầu Warshall); (d) test ngẫu nhiên có seed.

### 12.1. Oracle độc lập và test cố định

Oracle: dựng ma trận quan hệ, lấy **bao đóng bắc cầu** bằng Warshall rồi so từng vị trí. Không dùng Union-Find hay duyệt đồ thị.

In [20]:
def oracle_warshall(sentence1, sentence2, pairs):
    if len(sentence1) != len(sentence2):
        return False
    vocab = sorted({w for p in pairs for w in p} | set(sentence1) | set(sentence2))
    idx, rel = closure_relation(pairs, vocab)
    return all(rel[idx[a]][idx[b]] for a, b in zip(sentence1, sentence2))


solvers = {
    "dfs": similar_dfs,
    "bfs_label": similar_bfs_label,
    "union_find": similar_union_find,
    "merge_sets": similar_merge_sets,
    "submit": lambda s1, s2, p: Solution().areSentencesSimilarTwo(list(s1), list(s2), [list(x) for x in p]),
}

fixed_cases = [
    (examples[0][0], True),
    (examples[1][0], True),
    (examples[2][0], False),
    ((["a"], ["a"], []), True),                                  # bằng nhau, không có cặp
    ((["a"], ["b"], []), False),                                 # khác nhau, không có cặp
    ((["a", "b"], ["a"], [["a", "b"]]), False),                  # khác độ dài
    ((["a"], ["b"], [["a", "b"]]), True),                        # cặp trực tiếp
    ((["a"], ["b"], [["b", "a"]]), True),                        # chiều ngược
    ((["a"], ["c"], [["a", "b"], ["b", "c"]]), True),            # bắc cầu 2 bước
    ((["a"], ["d"], [["a", "b"], ["c", "d"]]), False),           # hai nhóm khác nhau
    ((["A"], ["a"], []), False),                                 # phân biệt hoa/thường
    ((["A"], ["a"], [["A", "a"]]), True),
    ((["x"], ["x"], [["a", "b"]]), True),                        # từ không có trong cặp nhưng bằng nhau
    ((["a"], ["a"], [["a", "a"]]), True),                        # cặp tự thân
    ((["a", "b"], ["b", "a"], [["a", "b"]]), True),
    ((["a", "b"], ["c", "c"], [["a", "c"], ["b", "c"]]), True),  # hai từ cùng nhóm với c
    ((["a", "b"], ["c", "d"], [["a", "c"]]), False),             # vị trí thứ hai không giống
]
for (s1, s2, pairs), expected in fixed_cases:
    assert oracle_warshall(s1, s2, pairs) == expected, ("oracle", s1, s2, pairs)
    for name, f in solvers.items():
        got = f(list(s1), list(s2), [list(p) for p in pairs])
        assert got == expected, (name, s1, s2, pairs, got, expected)
print("Test cố định: OK (", len(fixed_cases), "bộ x", len(solvers), "lời giải + oracle )")

Test cố định: OK ( 17 bộ x 5 lời giải + oracle )


### 12.2. Test chuỗi dài (kiểm tra `RecursionError`)

Chuỗi `w0 - w1 - ... - w2000` tạo cây Union-Find rất sâu nếu không nén đường đi / không hợp theo kích thước. Với `find` đệ quy và cách hợp `parent[find(a)] = find(b)`, độ sâu có thể tới ~2000, vượt giới hạn đệ quy mặc định (`sys.getrecursionlimit()` thường là 1000). Các cách của ta dùng vòng lặp nên an toàn.

In [21]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
chain_pairs = [["w%d" % i, "w%d" % (i + 1)] for i in range(2000)]
assert similar_union_find(["w0"], ["w2000"], chain_pairs) is True
assert similar_union_find(["w0"], ["x"], chain_pairs) is False
for name, f in solvers.items():
    assert f(["w0", "w1000"], ["w2000", "w1"], chain_pairs) is True, name
    assert f(["w0"], ["other"], chain_pairs) is False, name
print("Chuỗi 2000 cặp: mọi cách đều đúng, không RecursionError")

giới hạn đệ quy mặc định: 1000
Chuỗi 2000 cặp: mọi cách đều đúng, không RecursionError


### 12.3. Phản ví dụ cho các heuristic sai

- *"Chỉ tra cặp trực tiếp"* (không bắc cầu, như bài Sentence Similarity I): ví dụ 1 và 2 của đề sẽ trả sai.
- *"Chỉ gộp theo thứ tự đọc, `parent[a] = b`"* (không dùng `find` cho gốc): hai nhóm lớn gộp sai, mất liên thông. Mục 13 có ví dụ cụ thể.

In [22]:
def wrong_direct_only(sentence1, sentence2, pairs):
    if len(sentence1) != len(sentence2):
        return False
    direct = {(a, b) for a, b in pairs} | {(b, a) for a, b in pairs}
    return all(a == b or (a, b) in direct for a, b in zip(sentence1, sentence2))

for (s1, s2, pairs), expected in examples:
    got = wrong_direct_only(s1, s2, pairs)
    print("chỉ cặp trực tiếp:", got, "| đúng:", expected)
assert wrong_direct_only(*examples[0][0]) is False and examples[0][1] is True
assert wrong_direct_only(*examples[1][0]) is False and examples[1][1] is True

chỉ cặp trực tiếp: False | đúng: True
chỉ cặp trực tiếp: False | đúng: True
chỉ cặp trực tiếp: False | đúng: False


### 12.4. Đối chiếu oracle và test ngẫu nhiên có seed

Nhóm 1: từ vựng nhỏ (3–7 từ, nhiều va chạm). Nhóm 2: cỡ lớn đúng ràng buộc, so các cách với nhau.

In [23]:
rng = random.Random(737)
for _ in range(3000):
    vocab = [chr(97 + i) for i in range(rng.randint(1, 7))] + ["Z"]
    pairs = [[rng.choice(vocab), rng.choice(vocab)] for _ in range(rng.randint(0, 6))]
    n1 = rng.randint(1, 4)
    n2 = n1 if rng.random() < 0.9 else rng.randint(1, 4)
    s1 = [rng.choice(vocab) for _ in range(n1)]
    s2 = [rng.choice(vocab) for _ in range(n2)]
    expected = oracle_warshall(s1, s2, pairs)
    for name, f in solvers.items():
        assert f(s1, s2, pairs) == expected, (name, s1, s2, pairs)
print("Ngẫu nhiên cỡ nhỏ (seed 737): 3000 bộ, khớp oracle")

Ngẫu nhiên cỡ nhỏ (seed 737): 3000 bộ, khớp oracle


In [24]:
rng = random.Random(2026)
yes = no = 0
for case in range(80):
    V = rng.choice([30, 300, 4000])
    vocab = ["".join(rng.choice("abcXYZ") for _ in range(rng.randint(1, 6))) + str(i) for i in range(V)]
    P = rng.randint(0, 2000)
    pairs = [[rng.choice(vocab), rng.choice(vocab)] for _ in range(P)]
    n = rng.randint(1, 1000)
    # câu 2 sinh bằng cách thay mỗi từ của câu 1 bởi một từ cùng nhóm (nếu có) hoặc chính nó -> nhiều ca True
    s1 = [rng.choice(vocab) for _ in range(n)]
    s2 = list(s1)
    for i in range(n):
        if rng.random() < 0.3:
            s2[i] = rng.choice(vocab)
    res = {name: f(s1, s2, pairs) for name, f in solvers.items()}
    assert len(set(res.values())) == 1, (case, res)
    yes += res["union_find"]
    no += not res["union_find"]
print("Ngẫu nhiên cỡ lớn (seed 2026): 80 bộ khớp nhau; True:", yes, "| False:", no)

Ngẫu nhiên cỡ lớn (seed 2026): 80 bộ khớp nhau; True: 42 | False: 38


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi giống nghĩa là không bắc cầu

Chỉ tra tập cặp trực tiếp bỏ qua bắc cầu: trả `False` cho ví dụ 1 và 2 (đúng là `True`). Đây là lời giải của bài Sentence Similarity **I**, không phải **II**.

In [25]:
(s1, s2, pairs), expected = examples[1]
print("sai (chỉ cặp trực tiếp):", wrong_direct_only(s1, s2, pairs), "| đúng:", Solution().areSentencesSimilarTwo(s1, s2, pairs))
assert Solution().areSentencesSimilarTwo(s1, s2, pairs) is True

sai (chỉ cặp trực tiếp): False | đúng: True


### Lỗi 2 — Quên trường hợp hai từ bằng nhau nhưng không có trong cặp nào

Nếu chỉ kiểm tra qua `ids` / `find` thì từ không xuất hiện trong các cặp (như `"I"` và `"love"` ở ví dụ 2) bị coi là không giống chính nó → `KeyError` hoặc `False`. Phải kiểm tra `a == b` trước.

In [26]:
def wrong_no_equal_check(sentence1, sentence2, pairs):
    if len(sentence1) != len(sentence2):
        return False
    ids = {}
    for a, b in pairs:
        for w in (a, b):
            ids.setdefault(w, len(ids))
    parent = list(range(len(ids)))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    for a, b in pairs:
        parent[find(ids[a])] = find(ids[b])
    for a, b in zip(sentence1, sentence2):          # SAI: thiếu "if a == b: continue"
        if a not in ids or b not in ids or find(ids[a]) != find(ids[b]):
            return False
    return True

print("sai:", wrong_no_equal_check(["x"], ["x"], [["a", "b"]]), "| đúng:", Solution().areSentencesSimilarTwo(["x"], ["x"], [["a", "b"]]))
assert wrong_no_equal_check(["x"], ["x"], [["a", "b"]]) is False
assert Solution().areSentencesSimilarTwo(["x"], ["x"], [["a", "b"]]) is True

sai: False | đúng: True


### Lỗi 3 — Quên kiểm tra độ dài hai câu

`zip` dừng ở câu ngắn hơn, nên `["a","b"]` và `["a"]` có thể bị coi là giống nhau nếu không kiểm tra `len` trước.

In [27]:
def wrong_no_length_check(sentence1, sentence2, pairs):
    return all(a == b for a, b in zip(sentence1, sentence2))   # SAI: thiếu so độ dài (và thiếu bắc cầu, chỉ minh họa zip)

print("sai:", wrong_no_length_check(["a", "b"], ["a"], []), "| đúng:", Solution().areSentencesSimilarTwo(["a", "b"], ["a"], []))
assert wrong_no_length_check(["a", "b"], ["a"], []) is True
assert Solution().areSentencesSimilarTwo(["a", "b"], ["a"], []) is False

sai: True | đúng: False


### Lỗi 4 — So sánh `parent[x]` thay vì `find(x)`

`parent[x]` chỉ là cha trực tiếp, không phải gốc của nhóm. Hai từ cùng nhóm có thể có `parent` khác nhau. Ví dụ chuỗi `a-b`, `b-c`: sau các phép hợp, `parent[a]` có thể là `b` còn `parent[c]` là `c`.

In [28]:
def wrong_compare_parent(sentence1, sentence2, pairs):
    if len(sentence1) != len(sentence2):
        return False
    ids = {}
    for a, b in pairs:
        for w in (a, b):
            ids.setdefault(w, len(ids))
    parent = list(range(len(ids)))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    for a, b in pairs:
        parent[find(ids[a])] = find(ids[b])
    return all(a == b or (a in ids and b in ids and parent[ids[a]] == parent[ids[b]])   # SAI: dùng parent thay vì find
               for a, b in zip(sentence1, sentence2))

pairs_chain = [["a", "b"], ["b", "c"]]
print("sai:", wrong_compare_parent(["a"], ["c"], pairs_chain), "| đúng:", Solution().areSentencesSimilarTwo(["a"], ["c"], pairs_chain))
assert wrong_compare_parent(["a"], ["c"], pairs_chain) is False
assert Solution().areSentencesSimilarTwo(["a"], ["c"], pairs_chain) is True

sai: False | đúng: True


### Lỗi 5 — `find` đệ quy bị `RecursionError` với chuỗi dài

Với `find` đệ quy chuẩn và `parent[find(a)] = find(b)`, chuỗi `w0-w1, w1-w2, ...` tạo cây độ sâu ~2000 (vượt giới hạn đệ quy 1000 của Python). Giải pháp: viết `find` bằng vòng lặp (như code nộp), hoặc hợp theo kích thước.

In [29]:
def wrong_recursive_find(sentence1, sentence2, pairs):
    ids = {}
    for a, b in pairs:
        for w in (a, b):
            ids.setdefault(w, len(ids))
    parent = list(range(len(ids)))
    def find(x):                                   # đệ quy
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    for a, b in pairs:
        parent[find(ids[a])] = find(ids[b])
    return find(ids[sentence1[0]]) == find(ids[sentence2[0]])

try:
    wrong_recursive_find(["w0"], ["w2000"], chain_pairs)
    print("không lỗi (môi trường có giới hạn đệ quy cao)")
except RecursionError as e:
    print("RecursionError:", type(e).__name__)
print("đúng (lặp):", Solution().areSentencesSimilarTwo(["w0"], ["w2000"], chain_pairs))

RecursionError: RecursionError
đúng (lặp): True


## 14. Mẹo & nhận dạng pattern

- **"Quan hệ bắc cầu giữa các phần tử, hỏi hai phần tử có liên quan không"** → **Union-Find** hoặc thành phần liên thông (BFS/DFS). Hãy nhớ: quan hệ tương đương = chia lớp.
- **Gộp nhóm dần dần theo cặp** rất hợp Union-Find; nếu chỉ cần trả lời sau khi biết hết các cặp, BFS gán nhãn cũng gọn.
- **Chuỗi làm khóa**: ánh xạ chuỗi sang số nguyên (`ids`) để dùng mảng `parent`, hoặc dùng luôn `dict` làm `parent`.
- **Từ không có trong các cặp** → chỉ giống chính nó: nhớ nhánh `a == b`.
- **`find` lặp** (không đệ quy) để tránh `RecursionError` ở Python; nén đường đi + hợp theo kích thước cho gần `O(1)`.
- Các bài cùng họ: Sentence Similarity (734), Accounts Merge (721), Number of Provinces (547), Evaluate Division (399), Redundant Connection (684), Satisfiability of Equality Equations (990).
- Kiểm chứng bằng **Warshall** (bao đóng bắc cầu) trên đồ thị nhỏ là oracle độc lập tốt.

## 15. Biến thể

### 15.1. Giống nghĩa **không** bắc cầu (Sentence Similarity I, LeetCode 734)

Chỉ cần tập cặp trực tiếp (hai chiều) và so bằng nhau. Đây chính là `wrong_direct_only` ở trên, nhưng đúng cho bài 734.

In [30]:
def similar_one(sentence1, sentence2, pairs):
    if len(sentence1) != len(sentence2):
        return False
    direct = {(a, b) for a, b in pairs} | {(b, a) for a, b in pairs}
    return all(a == b or (a, b) in direct for a, b in zip(sentence1, sentence2))

print(similar_one(["great", "acting"], ["fine", "drama"], [["great", "fine"], ["drama", "acting"], ["skills", "talent"]]))   # True
print(similar_one(["a"], ["c"], [["a", "b"], ["b", "c"]]))                                                                 # False (không bắc cầu)
assert similar_one(["a"], ["c"], [["a", "b"], ["b", "c"]]) is False

True
False


### 15.2. Liệt kê các nhóm từ giống nghĩa

Nếu đề yêu cầu in các nhóm: duyệt mọi từ, nhóm theo `find`. Kết quả là các lớp tương đương.

In [31]:
def synonym_groups(pairs):
    ids = {}
    for a, b in pairs:
        for w in (a, b):
            ids.setdefault(w, len(ids))
    parent = list(range(len(ids)))

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    for a, b in pairs:
        ra, rb = find(ids[a]), find(ids[b])
        if ra != rb:
            parent[rb] = ra
    groups = defaultdict(list)
    for w, i in ids.items():
        groups[find(i)].append(w)
    return sorted(sorted(g) for g in groups.values())

print(synonym_groups(examples[0][0][2]))
print(synonym_groups(examples[1][0][2]))
assert synonym_groups(examples[0][0][2]) == [["acting", "drama"], ["fine", "good", "great"], ["skills", "talent"]]

[['acting', 'drama'], ['fine', 'good', 'great'], ['skills', 'talent']]
[['anime', 'leetcode', 'manga', 'onepiece', 'platform']]


### 15.3. Thêm cặp theo thời gian thực (online)

Nếu các cặp **đến dần** và giữa chừng có truy vấn "`x` có giống `y` không?", Union-Find là lựa chọn tự nhiên: mỗi cặp mới là một `union`, mỗi truy vấn là hai `find`. BFS gán nhãn thì phải chạy lại. Đây là lý do Union-Find hay được ưa chuộng cho các bài kiểu "hệ thống nhóm động".

### 15.4. Các biến thể khác (chỉ mô tả)

- **Có cặp "trái nghĩa"**: thêm ràng buộc xung đột; dùng Union-Find kèm "nhóm đối lập" (bài Possible Bipartition, 886).
- **Tính tương đồng có trọng số/độ tin cậy**: đồ thị có trọng số, tìm đường tốt nhất (Evaluate Division, 399, dùng Union-Find có trọng số).
- **Từ có nhiều nghĩa (đa nghĩa)**: quan hệ không bắc cầu hoàn toàn; cần mô hình khác.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS cho từng cặp từ | mỗi vị trí tìm đường từ `a` đến `b` | `O(n · (V + P))` | `O(V + P)` | dễ nghĩ; làm baseline/oracle |
| 2. BFS gán nhãn | gán mã nhóm cho mọi thành phần một lần | `O(V + P + n)` | `O(V + P)` | khi đã quen duyệt đồ thị; không cần cấu trúc đặc biệt |
| 3. Union-Find | hợp nhóm theo từng cặp, so `find` | `O((P + n) α(V))` | `O(V)` | **cách nên nộp**; ngắn, nhanh, hợp bài tập động |
| 4. Gộp tập nhỏ vào lớn | mỗi từ trỏ tới tập nhóm; gộp nhỏ vào lớn | `O(P log V + n)` | `O(V)` | khi muốn dễ viết, không cần `find` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
ids = {}
for w in ["b", "a", "b", "c"]:
    ids.setdefault(w, len(ids))
print(ids)
print(list(zip(["x", "y", "z"], [1, 2])))
print("A" == "a", "a" in ids, "d" in ids)
```

In [32]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `similarPairs = [["a","b"],["c","d"],["b","e"],["d","f"],["e","c"]]`: hãy lập các nhóm từ giống nghĩa. Với câu `["a","f","g"]` và `["d","a","g"]`, cho biết hai câu có giống nhau không.

In [33]:
# Bài 2: điền kết quả tính tay
hand_groups = None      # danh sách các nhóm, mỗi nhóm là list đã sắp xếp
hand_answer = None      # True / False

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành Union-Find (hàm `find` lặp có nén đường đi, và `union` theo kích thước).

In [34]:
class DSU:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x: int) -> int:
        # TODO 1: đi lên theo parent tới gốc root
        # TODO 2: nén đường đi: gắn mọi nút trên đường vào root
        return None

    def union(self, a: int, b: int) -> bool:
        # TODO 3: ra, rb = gốc của a, b; nếu bằng nhau trả False
        # TODO 4: gắn gốc có size nhỏ vào gốc có size lớn, cộng size; trả True
        return None


# d = DSU(5)
# assert d.union(0, 1) and d.union(1, 2) and not d.union(0, 2)
# assert d.find(0) == d.find(2) and d.find(0) != d.find(3)

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 2, hãy tự viết `exercise_4_bfs_label(sentence1, sentence2, pairs)`: dựng danh sách kề, gán nhãn thành phần bằng BFS rồi so từng vị trí.

In [35]:
def exercise_4_bfs_label(sentence1: List[str], sentence2: List[str], pairs: List[List[str]]) -> bool:
    # TODO
    pass


# assert exercise_4_bfs_label(*examples[0][0]) is True
# assert exercise_4_bfs_label(*examples[2][0]) is False
# assert exercise_4_bfs_label(["a"], ["a"], []) is True

### Bài 5 — Mini challenge: số nhóm và nhóm lớn nhất (khó)

Cho `pairs` (không có câu). Viết `exercise_5_stats(pairs)` trả về cặp `(số nhóm từ giống nghĩa, kích thước nhóm lớn nhất)`. Nếu `pairs` rỗng trả `(0, 0)`.

```python
exercise_5_stats([["a","b"],["b","c"],["d","e"]]) -> (2, 3)
exercise_5_stats([["a","a"]]) -> (1, 1)
exercise_5_stats([]) -> (0, 0)
```

Gợi ý: Union-Find có mảng `size`; hoặc dùng `synonym_groups` (mục 15) nhưng hãy tự viết lại bằng cách hợp theo kích thước.

In [36]:
def exercise_5_stats(pairs: List[List[str]]):
    # TODO
    pass


# assert exercise_5_stats([["a", "b"], ["b", "c"], ["d", "e"]]) == (2, 3)
# assert exercise_5_stats([["a", "a"]]) == (1, 1)
# assert exercise_5_stats([]) == (0, 0)

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `setdefault(k, v)` chỉ đặt khi khóa chưa có; `zip` dừng ở danh sách ngắn hơn.
- Mức 2: `ids` lần lượt gặp `b, a, c` (lần thứ hai của `b` bị bỏ qua).
- Mức 3: dòng 1 `{'b': 0, 'a': 1, 'c': 2}`; dòng 2 `[('x', 1), ('y', 2)]`; dòng 3 `False True False`.

**Bài 2**
- Mức 1: vẽ đồ thị: `a-b`, `b-e`, `e-c`, `c-d`, `d-f`: một chuỗi dài.
- Mức 2: các từ `a, b, c, d, e, f` đều nối với nhau thành một nhóm; `g` không có trong cặp.
- Mức 3: nhóm `[["a","b","c","d","e","f"]]`; vị trí 1: `a ~ d` (cùng nhóm), vị trí 2: `f ~ a` (cùng nhóm), vị trí 3: `g == g` → đáp án `True`.

**Bài 3**
- Mức 1: `root = x; while self.parent[root] != root: root = self.parent[root]`.
- Mức 2: nén: `while self.parent[x] != root: self.parent[x], x = root, self.parent[x]`.
- Mức 3: `union`: `if self.size[ra] < self.size[rb]: ra, rb = rb, ra`; `self.parent[rb] = ra`; `self.size[ra] += self.size[rb]`.

**Bài 4**
- Mức 1: kiểm tra độ dài trước; dựng `adj` bằng `defaultdict(list)`.
- Mức 2: với mỗi từ `start` trong `adj` chưa có nhãn, BFS bằng `deque` gán chung `cid`.
- Mức 3: so sánh `a == b or (a in comp and b in comp and comp[a] == comp[b])` cho mọi vị trí.

**Bài 5**
- Mức 1: gán `ids` cho các từ, rồi `DSU` với `size`.
- Mức 2: số nhóm = số gốc phân biệt (`find(i) == i`); nhóm lớn nhất = `max(size[gốc])`.
- Mức 3: nhớ nhánh `pairs` rỗng trả `(0, 0)`; cặp `["a","a"]` cho một nhóm size 1.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
ids = {}
for w in ["b", "a", "b", "c"]:
    ids.setdefault(w, len(ids))
print(ids)
print(list(zip(["x", "y", "z"], [1, 2])))
print("A" == "a", "a" in ids, "d" in ids)

assert ids == {"b": 0, "a": 1, "c": 2}
assert list(zip(["x", "y", "z"], [1, 2])) == [("x", 1), ("y", 2)]
assert ("A" == "a", "a" in ids, "d" in ids) == (False, True, False)
print("Bài 1 OK")

{'b': 0, 'a': 1, 'c': 2}
[('x', 1), ('y', 2)]
False True False
Bài 1 OK


### Đáp án Bài 2

Đồ thị: `a-b`, `b-e`, `e-c`, `c-d`, `d-f` nối thành một nhóm `{a, b, c, d, e, f}`. Vị trí 1: `a ~ d`; vị trí 2: `f ~ a`; vị trí 3: `g == g`. Đáp án `True`.

In [38]:
pairs2 = [["a", "b"], ["c", "d"], ["b", "e"], ["d", "f"], ["e", "c"]]
groups2 = synonym_groups(pairs2)
ans2 = Solution().areSentencesSimilarTwo(["a", "f", "g"], ["d", "a", "g"], pairs2)
print(groups2, ans2)

assert groups2 == [["a", "b", "c", "d", "e", "f"]]
assert ans2 is True
assert oracle_warshall(["a", "f", "g"], ["d", "a", "g"], pairs2) is True
print("Bài 2 OK")

[['a', 'b', 'c', 'd', 'e', 'f']] True
Bài 2 OK


### Đáp án Bài 3

In [39]:
class DSUSolution:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x: int) -> int:
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a: int, b: int) -> bool:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


d = DSUSolution(5)
assert d.union(0, 1) and d.union(1, 2) and not d.union(0, 2)
assert d.find(0) == d.find(2) and d.find(0) != d.find(3)
big = DSUSolution(5000)                         # chuỗi dài: find lặp không bị RecursionError
for i in range(4999):
    assert big.union(i, i + 1)
assert big.find(0) == big.find(4999)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_solution(sentence1: List[str], sentence2: List[str], pairs: List[List[str]]) -> bool:
    if len(sentence1) != len(sentence2):
        return False
    adj = defaultdict(list)
    for a, b in pairs:
        adj[a].append(b)
        adj[b].append(a)
    comp = {}
    cid = 0
    for start in adj:
        if start in comp:
            continue
        comp[start] = cid
        queue = deque([start])
        while queue:
            u = queue.popleft()
            for v in adj[u]:
                if v not in comp:
                    comp[v] = cid
                    queue.append(v)
        cid += 1
    return all(a == b or (a in comp and b in comp and comp[a] == comp[b]) for a, b in zip(sentence1, sentence2))


assert exercise_4_solution(*examples[0][0]) is True
assert exercise_4_solution(*examples[2][0]) is False
assert exercise_4_solution(["a"], ["a"], []) is True
rng = random.Random(4)
for _ in range(800):
    vocab = [chr(97 + i) for i in range(rng.randint(1, 6))]
    pairs = [[rng.choice(vocab), rng.choice(vocab)] for _ in range(rng.randint(0, 5))]
    s1 = [rng.choice(vocab) for _ in range(3)]
    s2 = [rng.choice(vocab) for _ in range(3)]
    assert exercise_4_solution(s1, s2, pairs) == oracle_warshall(s1, s2, pairs)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_solution(pairs: List[List[str]]):
    ids = {}
    for a, b in pairs:
        for w in (a, b):
            ids.setdefault(w, len(ids))
    if not ids:
        return (0, 0)
    d = DSUSolution(len(ids))
    for a, b in pairs:
        d.union(ids[a], ids[b])
    roots = {d.find(i) for i in range(len(ids))}
    return (len(roots), max(d.size[r] for r in roots))


assert exercise_5_solution([["a", "b"], ["b", "c"], ["d", "e"]]) == (2, 3)
assert exercise_5_solution([["a", "a"]]) == (1, 1)
assert exercise_5_solution([]) == (0, 0)
rng = random.Random(5)
for _ in range(500):
    vocab = [chr(97 + i) for i in range(rng.randint(1, 8))]
    pairs = [[rng.choice(vocab), rng.choice(vocab)] for _ in range(rng.randint(0, 7))]
    groups = synonym_groups(pairs)
    expected = (len(groups), max((len(g) for g in groups), default=0))
    assert exercise_5_solution(pairs) == expected, pairs
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def areSentencesSimilarTwo(self, sentence1: List[str], sentence2: List[str], similarPairs: List[List[str]]) -> bool:
        if len(sentence1) != len(sentence2):
            return False
        ids = {}
        for a, b in similarPairs:
            for w in (a, b):
                if w not in ids:
                    ids[w] = len(ids)
        parent = list(range(len(ids)))

        def find(x):                          # lặp + nén đường đi
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:
                parent[x], x = root, parent[x]
            return root

        for a, b in similarPairs:
            parent[find(ids[a])] = find(ids[b])
        return all(a == b or (a in ids and b in ids and find(ids[a]) == find(ids[b]))
                   for a, b in zip(sentence1, sentence2))
```

Câu thần chú: **"giống nghĩa là quan hệ tương đương: hợp nhóm theo cặp, rồi so `a == b` hoặc cùng gốc."**

### Checklist trước khi submit

- [ ] Kiểm tra `len(sentence1) == len(sentence2)` **trước** khi `zip`.
- [ ] Có nhánh `a == b` (từ không nằm trong cặp nào vẫn giống chính nó).
- [ ] Nếu `a` hoặc `b` không có trong `ids` (và `a != b`) thì trả `False`, không để `KeyError`.
- [ ] So sánh **`find(x)`**, không phải `parent[x]`.
- [ ] `find` viết **lặp** (hoặc nén đường đi + hợp theo kích thước), tránh `RecursionError` với chuỗi 2000 cặp.
- [ ] Phân biệt hoa/thường (so sánh chuỗi nguyên văn).
- [ ] Bài Premium: nếu có tài khoản, đối chiếu lại đề gốc và ví dụ.
- [ ] Skeleton đúng: `class Solution`, method `areSentencesSimilarTwo(self, sentence1, sentence2, similarPairs) -> bool`.